# Host Orchestrator Playground (Minimal)

Minimal wrapper to test the orchestrator with mock songs.
Set `current_song` and `next_song`, run the last cell, and the generated script is printed.

In [1]:
from __future__ import annotations

import random
import sys
import time
from pathlib import Path
from datetime import datetime
from IPython.display import Audio

# Resolve repo root and import from src/
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "src").exists():
            REPO_ROOT = parent
            break

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from neuralcast.pipelines.host_orchestrator_generation import (
    build_prompt,
    build_system_prompt,
    build_tts_instructions,
    generate_archetype_script,
    resolve_station_personality,
    station_name_for_generation,
)
from neuralcast.pipelines.host_orchestrator_models import (
    Archetype,
    QueueTrack,
    TrackMetadata,
)
from neuralcast.pipelines.host_orchestrator_state import (
    assemble_banned_list,
    choose_angle,
    choose_hook,
    default_state,
)
from neuralcast.services.openai_client import synthesize_speech

print(f"Repo root: {REPO_ROOT}")


Repo root: /home/nicou/Dropbox/Documents/Projects_and_Coding/Media_and_Content/NeuralCast


In [2]:
def _make_track(queue_id: str, song: dict[str, str]) -> QueueTrack:
    return QueueTrack(
        queue_id=queue_id,
        song_id=None,
        artist=str(song.get("artist", "")).strip(),
        title=str(song.get("title", "")).strip(),
        duration=None,
        raw={},
    )


def generate_script_for_tracks(
    current_song: dict[str, str],
    next_song: dict[str, str],
    archetype: Archetype = Archetype.BACK_SELL,
    station_slug: str = "neuralcast",
    station_display_name: str = "NeuralCast",
    current_meta: TrackMetadata | None = None,
    next_meta: TrackMetadata | None = None,
    seed: int = 42,
    angle_override: str | None = None,
    hook_override: str | None = None,
) -> tuple[str, str, str]:
    rng = random.Random(seed)
    state = default_state(time.time(), random.Random(seed + 1))

    current_track = _make_track("current", current_song)
    next_track = _make_track("next", next_song)

    personality = resolve_station_personality(station_slug)
    station_name = station_name_for_generation(station_slug, station_display_name)

    angle = (
        angle_override
        if angle_override is not None
        else choose_angle(archetype, state, rng)
    )
    hook = (
        hook_override
        if hook_override is not None
        else choose_hook(archetype, state, rng)
    )

    current_meta = current_meta or TrackMetadata()
    next_meta = next_meta or TrackMetadata()
    banned_list = assemble_banned_list(state)

    system_prompt = build_system_prompt(station_name, personality)
    user_prompt = build_prompt(
        archetype=archetype,
        station_name=station_name,
        personality=personality,
        current=current_track,
        next_track=next_track,
        current_meta=current_meta,
        next_meta=next_meta,
        angle=angle,
        hook=hook,
        banned_list=banned_list,
        recent_scripts=state.recent_scripts,
        schedule_context=None,
    )

    script_text, _news_segment, archetype_used = generate_archetype_script(
        archetype=archetype,
        station_name=station_name,
        personality=personality,
        current_track=current_track,
        next_track=next_track,
        current_meta=current_meta,
        next_meta=next_meta,
        angle=angle,
        hook=hook,
        banned_list=banned_list,
        schedule_context=None,
        state=state,
        rng=rng,
        forced_mode=(archetype == Archetype.NEWS),
    )

    print(f"archetype selected: {archetype_used.value}")
    print(f"angle: {angle}")
    print(f"hook: {hook}")
    print("-" * 80)
    return script_text, system_prompt, user_prompt


In [4]:
# Inputs
station_slug = "neuralforge"
station_display_name = "NeuralForge"
archetype = Archetype.BACK_SELL
seed = 42

current_song = {
    "artist": "Dream Theater",
    "title": "A Count of Tuscany",
}
next_song = {
    "artist": "Seventh Wonder",
    "title": "Tiara",
}

# Optional metadata (edit if useful, leave blank otherwise).
current_meta = TrackMetadata(year="2009", genre="Prog Metal")
next_meta = TrackMetadata(year="2018", genre="Prog Metal")

# Optional overrides (set to None to let orchestrator choose).
angle_override = None
hook_override = None

script, system_prompt, user_prompt = generate_script_for_tracks(
    current_song=current_song,
    next_song=next_song,
    archetype=archetype,
    station_slug=station_slug,
    station_display_name=station_display_name,
    current_meta=current_meta,
    next_meta=next_meta,
    seed=seed,
    angle_override=angle_override,
    hook_override=hook_override,
)

print(script)

archetype selected: back_sell
angle: Fanatic
hook: Cierra perfecto
--------------------------------------------------------------------------------
Ese cierre acústico de Dream Theater es impecable, te deja flotando después de semejante viaje técnico. La verdad que ahí la sensibilidad de Petrucci se nota en cada nota. Ahora seguimos en esa misma sintonía progresiva con Seventh Wonder; Tiara entra justo para sostener el pulso de este bloque.


In [6]:
print(system_prompt)

Sos la voz al aire de NéuralForsh. Acompanias a quien escucha entre tema y tema, sosteniendo el pulso del dia y el recorrido emocional de cada bloque con calidez, criterio y personalidad.

Objetivos centrales:
1) Sonar humano y variado durante toda la programacion.
2) Priorizar la musica y hablar con intencion clara.
3) Sumar contexto y clima sin inventar realidad.
4) Hacer que la persona que escucha se sienta acompanada en el momento.

Contrato de realidad:
- No afirmar presencia fisica/corporal, ubicacion de estudio ni manipulacion de objetos.
- No afirmar interacciones con oyentes salvo que haya datos explicitos.
- No afirmar experiencias personales del mundo real salvo que esten explicitamente provistas.
- No afirmar percepcion en tiempo real del entorno de quien escucha.

Estilo permitido:
- Tono de locutor radial confiado, con espontaneidad relajada.
- Se aceptan pequenas reflexiones de clima/animo cuando sean genericas y no factuales.
- Se aceptan metafora/humor si no implican h

In [7]:
print(user_prompt)

Estas generando un pase de cierre y puente.

Objetivo del arquetipo:
- Cerrar la cola emocional del tema que termino y guiar de forma fluida hacia el siguiente.
- Sonar como un pensamiento en vivo para una persona, no como anuncio para una multitud.

Como suena una buena salida:
- Conversacional, calida y segura.
- Especifica para el tema actual y el que viene.
- Nunca con tono slogan ni energia de saludo enlatado.
- La primera frase debe incluir una observacion musical concreta cuando los metadatos lo permitan.
- Evitar metaforas de musica pesada gastadas, salvo justificacion clara por contexto provisto.
- Forma sugerida: detalle sonoro concreto -> lectura corta -> puente al proximo tema.
- Dejar al menos una respiracion oral (pausa breve o muletilla suave) si entra natural.

Modo segun angulo:
- Minimalist: una observacion clara y pase limpio.
- Connector: puente musical/tematico real usando metadatos provistos.
- Fanatic: entusiasmo contenido, con control y brevedad.

Regla anti-rep

## Create TTS MP3

Uses the generated `script` and saves an MP3 locally (no radio/AzuraCast interaction).

In [10]:
tts_provider = "gemini"  # "gemini" or "openai"
gemini_tts_model = "gemini-2.5-flash-preview-tts"
gemini_tts_voice = "Enceladus"
openai_tts_model = "gpt-4o-mini-tts"
openai_tts_voice = "ash"

personality = resolve_station_personality(station_slug)
tts_instructions = build_tts_instructions(personality)

output_dir = (
    REPO_ROOT / "src/neuralcast/assets/stories/snippets/host_orchestrator_playground"
)
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
audio_path = output_dir / f"{station_slug}_{archetype.value}_{timestamp}.mp3"

synthesize_speech(
    text=script,
    outfile=str(audio_path),
    provider=tts_provider,
    instructions=tts_instructions,
    gemini_model=gemini_tts_model,
    gemini_voice=gemini_tts_voice,
    openai_model=openai_tts_model,
    openai_voice=openai_tts_voice,
)

print(f"Saved MP3: {audio_path}")
Audio(filename=str(audio_path))


Saved MP3: /home/nicou/Dropbox/Documents/Projects_and_Coding/Media_and_Content/NeuralCast/src/neuralcast/assets/stories/snippets/host_orchestrator_playground/neuralforge_deep_dive_2026-02-21_232745.mp3
